# ECG-image inference and Grad-CAM in Google Colab

Run individual models or their raw-logit ensemble with the paper's prediction defaults, and optionally display Grad-CAM for ViT and ConvNeXtV2. This notebook targets source release **v0.3.1** and the separately published **v0.2.0 weights**. The source-loading cell selects `v0.3.1`. On 2026-10-28, the matching candidate archive completed this workflow on Colab/A100 with filelock 4.2.1; final public-tag source acquisition is a separate post-publication check. The earlier `v0.3.0` public-tag check completed on 2026-10-24. Review the repository's weight terms before downloading; the complete bundle includes the ViT noncommercial condition.

Select a **GPU runtime**. Setup creates a separate inference environment with PyTorch 2.13.0 / torchvision 0.28.0 CUDA 13.0, transformers 5.10.4, Pillow 13.0.0 and filelock 4.2.1. Model and image code runs in that environment; the notebook kernel coordinates commands and displays progress.

The default input is a generated **480 x 480 synthetic trace** for checking software execution. To use an existing permitted ECG image, select `INPUT_SOURCE = "file"` and set `USER_IMAGE_PATH` in the input cell. Synthetic predictions and heatmaps have no clinical meaning. Save the prediction, its matching run record and optional visualization files using the final download cell.


## 1. Load the source release


In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata as metadata
import json
import os
import re
import subprocess
import sys

# A failed source-selection attempt must invalidate earlier success state.
ENVIRONMENT_READY = False
WEIGHTS_READY = False
result = None

if sys.version_info < (3, 11):
    raise RuntimeError("Select a Colab runtime with Python 3.11 or later.")

REPO_URL = "https://github.com/ekagawa007/takotsubo_ami_ai_ecg.git"
GIT_REF = "v0.3.1"  # Available after publication of this release.
REPO_DIR = Path("/content/takotsubo_ami_ai_ecg")
PACKAGE_SOURCE_COMMIT = globals().get("_VERIFIED_PACKAGE_SOURCE_COMMIT")

if not re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_./-]*", GIT_REF):
    raise ValueError("Set GIT_REF to a branch name, tag name, or commit SHA.")
if REPO_DIR.is_symlink():
    raise RuntimeError("REPO_DIR must not be a symbolic link.")

def git(*args):
    return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
else:
    if not (REPO_DIR / ".git").is_dir() or (REPO_DIR / ".git").is_symlink():
        raise RuntimeError("REPO_DIR is not a regular Git working directory. Choose another location.")
    if git("remote", "get-url", "origin").rstrip("/") != REPO_URL.rstrip("/"):
        raise RuntimeError("The repository in REPO_DIR does not match REPO_URL.")
    if git("status", "--porcelain"):
        raise RuntimeError("The working directory has changes. Save them before rerunning this cell.")

try:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth=1", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError("Could not load the requested source revision. Check the log and confirm GIT_REF is published.") from error
CODE_COMMIT = git("rev-parse", "HEAD")
if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise RuntimeError("Git did not return a complete commit identifier.")

required_names = [
    "requirements-inference.txt", "tools/download_weights.py",
    "src/takotsubo_ami_ai_ecg/checkpoint_compat.py",
    "src/takotsubo_ami_ai_ecg/gradcam.py",
    "src/takotsubo_ami_ai_ecg/security.py",
    "src/takotsubo_ami_ai_ecg/image_input.py",
    "src/takotsubo_ami_ai_ecg/safe_output.py",
    "src/takotsubo_ami_ai_ecg/progress.py",
    "src/takotsubo_ami_ai_ecg/assets/weights_v0.2.0.json",
    *[f"examples/predict_{name}.py" for name in ("ensemble", "vit", "convnextv2", "mlpmixer")],
]
required_names = sorted(set(required_names) | {
    p.relative_to(REPO_DIR).as_posix() for p in (REPO_DIR / "src/takotsubo_ami_ai_ecg").glob("*.py")
})
SOURCE_SHA256 = {}
for name in required_names:
    path = REPO_DIR / name
    if any(p.is_symlink() for p in (path, *path.parents)) or not path.is_file():
        raise RuntimeError("The selected source lacks a required regular inference file: " + name)
    SOURCE_SHA256[name] = hashlib.sha256(path.read_bytes()).hexdigest()

def verify_source():
    if REPO_DIR.is_symlink() or not REPO_DIR.is_dir() or git("rev-parse", "HEAD") != CODE_COMMIT:
        raise RuntimeError("The source checkout changed. Start a fresh runtime.")
    if git("status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError("Tracked source files changed. Save your edits and start a fresh runtime.")
    for name, expected in SOURCE_SHA256.items():
        path = REPO_DIR / name
        if any(p.is_symlink() for p in (path, *path.parents)) or not path.is_file():
            raise RuntimeError("A source file is missing or is a symbolic link.")
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise RuntimeError("An inference source file changed. Start a fresh runtime.")

for module_name, module in tuple(sys.modules.items()):
    if module_name == "takotsubo_ami_ai_ecg" or module_name.startswith("takotsubo_ami_ai_ecg."):
        module_file = getattr(module, "__file__", None)
        if (PACKAGE_SOURCE_COMMIT != CODE_COMMIT
                or module_file and not Path(module_file).resolve().is_relative_to(REPO_DIR.resolve())):
            raise RuntimeError("A different package revision is loaded. Start a fresh Colab runtime.")
verify_source()
# Keep this binding unchanged if a later source-selection attempt fails.
_VERIFIED_PACKAGE_SOURCE_COMMIT = CODE_COMMIT
sys.path.insert(0, str(REPO_DIR / "src"))
print("Code commit:", CODE_COMMIT)


## 2. Create the separate inference environment

Select a GPU in the runtime settings. Setup installs the tested CUDA 13.0 pair and inference requirements into `/content/ecg_inference_venv`. It can take several minutes. It does not replace the notebook kernel's ML packages, so a session restart is normally unnecessary.

After a manual restart, run from the first code cell; a completed environment is checked and reused. If installation fails partway, preserve the error log and start a fresh runtime. A partially installed environment is never treated as ready.


In [ ]:
ENVIRONMENT_READY = False
VENV_DIR = Path("/content/ecg_inference_venv")
INFERENCE_PYTHON = VENV_DIR / "bin" / "python"
READY_MARKER = VENV_DIR / "inference_ready.json"
EXPECTED_VERSIONS = {
    "torch": "2.13.0+cu130", "torchvision": "0.28.0+cu130",
    "transformers": "5.10.4", "timm": "1.0.22", "numpy": "2.2.6",
    "Pillow": "13.0.0", "safetensors": "0.7.0", "setuptools": "84.2.1", "pip": "26.2.2", "filelock": "4.2.1",
}
EXPECTED_SETUP = {"code_commit": CODE_COMMIT, "requirements_sha256": SOURCE_SHA256["requirements-inference.txt"], "versions": EXPECTED_VERSIONS}
if VENV_DIR.is_symlink():
    raise RuntimeError("The inference environment path must not be a symbolic link.")
if VENV_DIR.exists():
    if not READY_MARKER.is_file() or READY_MARKER.is_symlink():
        raise RuntimeError("The environment is incomplete. Save the error log and use a fresh Colab runtime.")
    if json.loads(READY_MARKER.read_text(encoding="utf-8")) != EXPECTED_SETUP:
        raise RuntimeError("This environment belongs to a different source revision. Use a fresh runtime.")
else:
    pip_version = tuple(int(v) for v in metadata.version("pip").split(".")[:2])
    if pip_version < (22, 3):
        raise RuntimeError("This setup needs pip 22.3 or later. Select a current Colab runtime.")
    VENV_DIR.mkdir(exist_ok=False)
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(VENV_DIR)], check=True)
    managed_pip = [sys.executable, "-m", "pip", "--python", str(INFERENCE_PYTHON)]
    subprocess.run(managed_pip + ["install", "pip==26.2.2", "setuptools==84.2.1", "--index-url", "https://pypi.org/simple"], check=True)
    subprocess.run(managed_pip + [
        "install", "torch==2.13.0", "torchvision==0.28.0",
        "--index-url", "https://download.pytorch.org/whl/cu130",
    ], check=True)
    subprocess.run(managed_pip + [
        "install", "-r", str(REPO_DIR / "requirements-inference.txt"),
        "--index-url", "https://pypi.org/simple",
    ], check=True)
    subprocess.run([str(INFERENCE_PYTHON), "-m", "pip", "check"], check=True)
    installed = json.loads(subprocess.check_output(
        [str(INFERENCE_PYTHON), "-c",
         "import importlib.metadata as m,json; print(json.dumps({k:m.version(k) for k in " + repr(tuple(EXPECTED_VERSIONS)) + "}))"],
        text=True,
    ))
    if installed != EXPECTED_VERSIONS:
        raise RuntimeError("Inference package versions differ from the reviewed versions.")
    from takotsubo_ami_ai_ecg.safe_output import write_files
    write_files([(READY_MARKER, (json.dumps(EXPECTED_SETUP, indent=2) + "\n").encode("utf-8"))])

subprocess.run([str(INFERENCE_PYTHON), "-m", "pip", "check"], check=True)
ENVIRONMENT_READY = True

def check_inference_environment():
    if not globals().get("ENVIRONMENT_READY", False):
        raise RuntimeError("Run the setup cells in order after opening or restarting the session.")
    if not INFERENCE_PYTHON.is_file() or READY_MARKER.is_symlink():
        raise RuntimeError("The inference environment is unavailable.")
    if json.loads(READY_MARKER.read_text(encoding="utf-8")) != EXPECTED_SETUP:
        raise RuntimeError("The inference environment marker changed.")
    verify_source()

check_inference_environment()
print("Inference environment ready. Notebook-kernel ML packages were not replaced.")
print("Inference interpreter:", INFERENCE_PYTHON)


In [ ]:
check_inference_environment()
runtime_versions = json.loads(subprocess.check_output(
    [str(INFERENCE_PYTHON), "-c", 'import importlib.metadata as metadata\nimport json\nimport sys\nimport torch\nimport torchvision\nfrom PIL import Image\nfrom transformers import ViTConfig, ViTForImageClassification\nversions = {"python": sys.version.split()[0]}\nfor package in ("torch", "torchvision", "timm", "transformers", "numpy", "Pillow", "safetensors", "setuptools", "pip", "filelock"):\n    versions[package] = metadata.version(package)\nversions["installed_packages"] = sorted([{ "name": d.metadata["Name"], "version": d.version } for d in metadata.distributions() if d.metadata.get("Name")], key=lambda item: item["name"].lower())\nversions["cuda_available"] = torch.cuda.is_available()\nversions["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None\nprint(json.dumps(versions))'], text=True,
))
if any(runtime_versions[name] != version for name, version in EXPECTED_VERSIONS.items()):
    raise RuntimeError("The inference interpreter has unexpected package versions.")
if not runtime_versions["cuda_available"]:
    raise RuntimeError("Inference CUDA is unavailable. Select a GPU runtime and inspect the setup log.")
print(json.dumps({k: v for k, v in runtime_versions.items() if k != "installed_packages"}, indent=2))
print("Recorded installed distributions:", len(runtime_versions["installed_packages"]))


## 3. Prepare the weights

The default downloads the public v0.2.0 weights and verifies the archive and every bundled file's SHA-256. The selected checkpoints are unchanged for this Grad-CAM extension. To use existing inference weights, set `WEIGHT_SOURCE = "local"` and edit `WEIGHTS_DIR`.

Set `USE_GOOGLE_DRIVE = True` below only if your image or local weights are on Drive. No Drive connection is needed for the synthetic example and automatic download.


In [ ]:
USE_GOOGLE_DRIVE = False  # Set to True only when reading files from Drive.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

In [ ]:
WEIGHTS_READY = False
check_inference_environment()

WEIGHT_SOURCE = "release"  # "release" for verified public weights; "local" for existing files.
WEIGHTS_DIR = Path("/content/ecg_inference_weights")
# Examples for WEIGHT_SOURCE = "local":
# WEIGHTS_DIR = Path("/content/weights")
# WEIGHTS_DIR = Path("/content/drive/MyDrive/ecg_inference/weights")

if WEIGHT_SOURCE not in {"release", "local"}:
    raise ValueError('Set WEIGHT_SOURCE to "release" or "local".')
if WEIGHT_SOURCE == "release":
    download_command = [
        str(INFERENCE_PYTHON), "-u", str(REPO_DIR / "tools" / "download_weights.py"),
        "--output-dir", str(WEIGHTS_DIR),
    ]
    download_process = None
    try:
        download_process = subprocess.Popen(
            download_command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in download_process.stdout:
            print(line, end="", flush=True)
        download_returncode = download_process.wait()
        if download_returncode != 0:
            raise RuntimeError(
                f"Weight download or verification failed with exit code {download_returncode}. "
                "Review the error above. Check the v0.2.0 Release asset or select existing local weights."
            )
    except BaseException:
        if download_process is not None and download_process.poll() is None:
            download_process.terminate()
            try:
                download_process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                download_process.kill()
                download_process.wait(timeout=5)
        raise
    finally:
        if download_process is not None and download_process.stdout is not None:
            download_process.stdout.close()
else:
    if not WEIGHTS_DIR.is_dir():
        raise FileNotFoundError("Set WEIGHTS_DIR to the existing inference-weight directory.")
    print("Using local inference weights; no download requested.")

WEIGHTS_READY = True
print("Weights directory:", WEIGHTS_DIR)


## 4. Choose the input image

Keep `INPUT_SOURCE = "synthetic"` for a software check. Select `"file"` and change `USER_IMAGE_PATH` to use an existing permitted image; that file is only read. Enable the Drive cell above when using a Drive path. Changing `USER_IMAGE_PATH` never redirects synthetic generation.

Supported inputs are single-frame PNG, JPEG, BMP, TIFF or WebP. Animated and multipage images are rejected. `strict480` (the default in the next cell) requires prepared **480 x 480 pixel** inputs. Explicit `resize` resizes the entire image bilinearly, without cropping or padding, and can change the aspect ratio.


In [ ]:
check_inference_environment()
INPUT_SOURCE = "synthetic"  # "synthetic" or "file".
USER_IMAGE_PATH = Path("/content/drive/MyDrive/ecg_inference/data/example.png")
SYNTHETIC_IMAGE_PATH = Path("/content/ecg_inference_synthetic.png")

if INPUT_SOURCE == "synthetic":
    IMAGE_PATH = SYNTHETIC_IMAGE_PATH
    input_script = 'from pathlib import Path\nimport hashlib, io, json, math, sys\nsys.path.insert(0, sys.argv[1])\nfrom PIL import Image, ImageDraw\nfrom takotsubo_ami_ai_ecg.security import require_patched_pillow\nfrom takotsubo_ami_ai_ecg.safe_output import write_files\nrequire_patched_pillow()\nimage = Image.new("RGB", (480, 480), "white")\ndraw = ImageDraw.Draw(image)\nfor position in range(0, 480, 12):\n    draw.line((position, 0, position, 479), fill=(255, 225, 225))\n    draw.line((0, position, 479, position), fill=(255, 225, 225))\nfor row in range(4):\n    points = [(x, 60 + row * 110 + int(8 * math.sin(x / 13) - (30 if x % 90 == 45 else 0))) for x in range(480)]\n    draw.line(points, fill=(35, 35, 35), width=2)\npixels_sha = hashlib.sha256(image.tobytes()).hexdigest()\nbuffer = io.BytesIO()\nimage.save(buffer, format="PNG")\nimage_bytes = buffer.getvalue()\npath = Path(sys.argv[2])\nif path.exists() or path.is_symlink():\n    if path.is_symlink() or path.read_bytes() != image_bytes:\n        raise RuntimeError("The synthetic input path contains different data.")\nelse:\n    write_files([(path, image_bytes)])\nprint(json.dumps({"kind": "generated_synthetic_trace", "size": [480,480], "mode": "RGB",\n                  "input_sha256": hashlib.sha256(image_bytes).hexdigest(),\n                  "rgb_pixel_sha256": pixels_sha}))'
elif INPUT_SOURCE == "file":
    IMAGE_PATH = USER_IMAGE_PATH
    if not IMAGE_PATH.is_file():
        raise FileNotFoundError("Set USER_IMAGE_PATH to an existing image; mount Drive above if needed.")
    input_script = 'import hashlib, json, sys\nfrom pathlib import Path\nsys.path.insert(0, sys.argv[1])\nfrom takotsubo_ami_ai_ecg.image_input import open_supported_image\npath = Path(sys.argv[2])\nwith open_supported_image(path, 1) as image:\n    image.load()\n    record = {"kind": "user_provided_image", "size": list(image.size), "mode": image.mode,\n              "rgb_pixel_sha256": hashlib.sha256(image.convert("RGB").tobytes()).hexdigest()}\nrecord["input_sha256"] = hashlib.sha256(path.read_bytes()).hexdigest()\nprint(json.dumps(record))\n'
else:
    raise ValueError('Set INPUT_SOURCE to "synthetic" or "file".')
input_record = json.loads(subprocess.check_output(
    [str(INFERENCE_PYTHON), "-c", input_script, str(REPO_DIR / "src"), str(IMAGE_PATH)], text=True,
))
print("Input:", json.dumps(input_record, indent=2))


## 5. Set inference options

These defaults match the paper. TTA counts include the unchanged view; `1` means no TTA. The ensemble averages raw logits and then applies softmax.

`ENABLE_GRADCAM = True` adds ViT and ConvNeXtV2 visualizations for the selected prediction model(s). Maps use one unaugmented float32 forward pass; they are **not TTA-averaged maps**. No MLP-Mixer or combined ensemble heatmap is produced. For MLP-Mixer alone, set `ENABLE_GRADCAM = False`.

The default target is each model's own TTA-predicted class. You can choose `ensemble-predicted` when `MODEL = "ensemble"`, or the explicit class `tako` / `AMI`. The target can differ between models. `CAM_NORMALIZATION = "inference"` matches the current CLI. The optional `legacy-vit` comparison mode uses mean/std 0.5 for ViT and does not explain the current ViT input preprocessing or establish historical-map reproduction.


In [ ]:
check_inference_environment()
MODEL = "ensemble"  # "ensemble", "vit", "convnextv2" or "mlpmixer".
OUTPUT_DIR = Path("/content/ecg_inference_results")
VIT_TTA = 24
VIT_MAGNITUDE = 1
CONVNEXTV2_TTA = 1
CONVNEXTV2_MAGNITUDE = 5  # Unused when TTA=1.
MLPMIXER_TTA = 32
MLPMIXER_MAGNITUDE = 3
ENSEMBLE_WEIGHTS = [1.0, 1.0, 1.0]
THRESHOLD = None
SEED = 1234
DEVICE = "cuda"
PRECISION = "float32"
INPUT_POLICY = "strict480"
BATCH_SIZE = 1
NUM_WORKERS = 4

ENABLE_GRADCAM = True
CAM_TARGET = "model-predicted"  # "model-predicted", "ensemble-predicted", "tako" or "AMI".
CAM_NORMALIZATION = "inference"  # "legacy-vit" is a comparison mode only.
CAM_OPACITY = 0.6  # 0 shows the input; 1 shows the color heatmap.


## 6. Check the inputs and run inference

`strict480` requires **480 x 480 pixel** inputs, matching the original dataset evaluation. If your images have a different size, check that the preprocessing matches training. `resize` is an explicit alternative and does not guarantee that the image preparation matches the paper.

The seed controls TTA randomness. However, results may not exactly match the original batch evaluation if image order, batch size, worker count, libraries, or numerical precision differ.
Each run of the inference cell creates a new output filename containing a timestamp and an identifier. Results with no TTA and with the paper's TTA settings can be saved as separate JSON files and run records.

The inference cell displays a live progress bar with overall completion, the current model, TTA view, image count, and elapsed time. Inference and Grad-CAM report separate progress stages. During inference, the percentage counts completed image/view evaluations; it is not a time estimate and may pause while models load or results are saved. Logs and errors appear as they arrive. Stopping the cell also stops its inference subprocess. The final success message appears after both the prediction and matching run record have been saved.


In [ ]:
check_inference_environment()
if not globals().get("WEIGHTS_READY", False):
    raise RuntimeError("Complete the weight-preparation cell successfully before checking inputs.")

supported_models = ("ensemble", "vit", "convnextv2", "mlpmixer")
if MODEL not in supported_models:
    raise ValueError(f"Choose MODEL from {supported_models}.")
if ENABLE_GRADCAM and MODEL == "mlpmixer":
    raise ValueError("Grad-CAM is implemented for ViT and ConvNeXtV2. Set ENABLE_GRADCAM = False for MLP-Mixer alone.")
if ENABLE_GRADCAM and CAM_TARGET == "ensemble-predicted" and MODEL != "ensemble":
    raise ValueError('CAM_TARGET = "ensemble-predicted" requires MODEL = "ensemble".')
if not IMAGE_PATH.is_file():
    raise FileNotFoundError("Provide the input image and set IMAGE_PATH to its location.")

selected_models = ("vit", "convnextv2", "mlpmixer") if MODEL == "ensemble" else (MODEL,)
required_weights = [WEIGHTS_DIR / model / "model.safetensors" for model in selected_models]
if "vit" in selected_models:
    required_weights.append(WEIGHTS_DIR / "vit" / "config.json")
if any(not path.is_file() for path in required_weights):
    raise FileNotFoundError("Inference weights or configuration files are missing for the selected model(s). Check the layout above.")

input_checked = json.loads(subprocess.check_output(
    [str(INFERENCE_PYTHON), "-c", 'import hashlib, json, sys\nfrom pathlib import Path\nsys.path.insert(0, sys.argv[1])\nfrom takotsubo_ami_ai_ecg.image_input import open_supported_image\npath = Path(sys.argv[2])\nwith open_supported_image(path, 1) as image:\n    size = list(image.size)\n    image.verify()\nprint(json.dumps({"size":size,"input_sha256":hashlib.sha256(path.read_bytes()).hexdigest()}))', str(REPO_DIR / "src"), str(IMAGE_PATH)], text=True,
))
input_size = tuple(input_checked["size"])
if INPUT_POLICY not in {"strict480", "resize"}:
    raise ValueError('Set INPUT_POLICY to "strict480" or "resize".')
if INPUT_POLICY == "strict480" and input_size != (480, 480):
    raise ValueError("strict480 requires 480 x 480 pixels. Prepare the image or explicitly select resize.")
if input_checked["input_sha256"] != input_record["input_sha256"]:
    raise RuntimeError("The input image changed after preparation. Run the input cell again.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input dimensions:", input_size)
print("Selected model:", MODEL)


In [ ]:
result = None

from datetime import datetime, timezone
import hashlib
from html import escape
import math
import time
import uuid
from IPython.display import HTML, display

check_inference_environment()
if not globals().get("WEIGHTS_READY", False):
    raise RuntimeError("Complete the weight-preparation cell successfully before running inference.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_JSON = OUTPUT_DIR / f"prediction_{MODEL}_{stamp}_{uuid.uuid4().hex[:8]}.json"
RUN_JSON = OUTPUT_JSON.with_suffix(".colab_run.json")

command = [
    str(INFERENCE_PYTHON), "-u",
    str(REPO_DIR / "examples" / f"predict_{MODEL}.py"),
    "--image", str(IMAGE_PATH),
    "--weights-dir", str(WEIGHTS_DIR),
    "--output", str(OUTPUT_JSON),
    "--vit-tta", str(VIT_TTA), "--vit-magnitude", str(VIT_MAGNITUDE),
    "--convnextv2-tta", str(CONVNEXTV2_TTA), "--convnextv2-magnitude", str(CONVNEXTV2_MAGNITUDE),
    "--mlpmixer-tta", str(MLPMIXER_TTA), "--mlpmixer-magnitude", str(MLPMIXER_MAGNITUDE),
    "--ensemble-weights", *[str(value) for value in ENSEMBLE_WEIGHTS],
    "--seed", str(SEED),
    "--device", DEVICE,
    "--precision", PRECISION,
    "--input-policy", INPUT_POLICY,
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--progress", "json",
]
if ENABLE_GRADCAM:
    command.extend(["--gradcam", "--cam-target", CAM_TARGET,
                    "--cam-normalization", CAM_NORMALIZATION, "--cam-opacity", str(CAM_OPACITY)])
if THRESHOLD is not None:
    command.extend(["--threshold", str(THRESHOLD)])

progress_prefix = "ECG_PROGRESS "
progress_state = {"completed": 0, "total": 0, "message": "Starting inference."}
progress_display = None
process = None
progress_error = False
started = time.monotonic()


def finite_number(value, default=0.0):
    try:
        number = float(value)
        return number if math.isfinite(number) else default
    except (TypeError, ValueError):
        return default


def count_pair(current, total):
    if current is None or total is None:
        return "n/a"
    return f"{escape(str(current))}/{escape(str(total))}"


def render_progress(message=None):
    global progress_display
    total = max(0.0, finite_number(progress_state.get("total")))
    completed = max(0.0, finite_number(progress_state.get("completed")))
    percent = min(100.0, 100.0 * completed / total) if total else 0.0
    elapsed = max(time.monotonic() - started, finite_number(progress_state.get("elapsed_seconds")))
    model = progress_state.get("model") or "n/a"
    model_position = count_pair(progress_state.get("model_index"), progress_state.get("model_count"))
    tta = count_pair(progress_state.get("tta_index"), progress_state.get("tta_total"))
    images = count_pair(progress_state.get("images_completed"), progress_state.get("images_total"))
    status = escape(str(message if message is not None else progress_state.get("message", "Running inference.")))
    content = HTML(
        '<div style="padding:10px 0;max-width:760px">'
        f'<div><strong>{status}</strong></div>'
        f'<progress value="{percent:.2f}" max="100" style="width:100%;height:20px" '
        'aria-label="Inference progress"></progress>'
        f'<div>{percent:.1f}% | Model: {escape(str(model))} ({model_position}) '
        f'| TTA: {tta} | Images: {images} | Elapsed: {elapsed:.1f} s</div></div>'
    )
    if progress_display is None:
        progress_display = display(content, display_id=True)
    else:
        progress_display.update(content)


def gradcam_artifact_paths(prediction_result):
    """Accept only documented files in this run's output directory."""
    artifacts = []
    for row in prediction_result["predictions"]:
        cam = row.get("gradcam")
        if cam is None:
            continue
        sample_id = row.get("sample_id", "")
        if not isinstance(sample_id, str) or not re.fullmatch(r"sample_[0-9]{6}", sample_id):
            raise RuntimeError("Unexpected visualization sample identifier.")
        stem = f"{OUTPUT_JSON.stem}_{sample_id}"
        if cam.get("input_image") != f"{stem}_input.png":
            raise RuntimeError("Unexpected visualization input filename.")
        names = [cam["input_image"]]
        selected = set(row["models"]) & {"vit", "convnextv2"}
        if set(cam.get("models", {})) != selected:
            raise RuntimeError("The visualization model set does not match the prediction.")
        for model in sorted(selected):
            expected = {"heatmap": f"{stem}_{model}_heatmap.png",
                        "overlay": f"{stem}_{model}_overlay.png",
                        "values": f"{stem}_{model}_cam.npy"}
            if cam["models"][model].get("files") != expected:
                raise RuntimeError("Unexpected visualization artifact filenames.")
            names.extend(expected.values())
        for name in names:
            path = OUTPUT_JSON.parent / name
            if any(p.is_symlink() for p in (path, *path.parents)) or not path.is_file():
                raise RuntimeError("A visualization artifact is missing or is a symbolic link.")
            if path.stat().st_size > 10 * 1024 * 1024:
                raise RuntimeError("A visualization artifact exceeds the expected size.")
            artifacts.append(path)
    if len({path.name for path in artifacts}) != len(artifacts):
        raise RuntimeError("Visualization artifact names are not unique.")
    return artifacts


def stop_inference_process():
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)


try:
    render_progress()
    process = subprocess.Popen(
        command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        if line.startswith(progress_prefix):
            try:
                event = json.loads(line[len(progress_prefix):])
            except json.JSONDecodeError:
                print("An unreadable progress update was skipped.", flush=True)
                continue
            if not isinstance(event, dict) or event.get("event") not in {"stage", "progress", "complete", "error"}:
                print("An unrecognized progress update was skipped.", flush=True)
                continue
            progress_state.update(event)
            if event["event"] == "error":
                progress_error = True
                render_progress("Inference failed. Review the logs below.")
            elif event["event"] == "complete":
                render_progress("Prediction saved; finishing the run record.")
            else:
                render_progress()
        else:
            print(line, end="", flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(
            f"Inference stopped with exit code {returncode}. Review the error displayed above."
        )
    if progress_error:
        raise RuntimeError("Inference reported an error. Review the logs above.")

    pending_result = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))

    if DEVICE == "cuda" and pending_result["metadata"]["actual_device"] != "cuda":
        raise RuntimeError("The inference did not use CUDA.")
    if any(pending_result["metadata"]["versions"][name] != version
           for name, version in EXPECTED_VERSIONS.items() if name not in {"setuptools", "pip"}):
        raise RuntimeError("Inference did not use the reviewed inference versions.")
    if len(pending_result["predictions"]) != 1 or pending_result["predictions"][0]["input_sha256"] != input_record["input_sha256"]:
        raise RuntimeError("Inference did not use the recorded input image.")
    if hashlib.sha256(IMAGE_PATH.read_bytes()).hexdigest() != input_record["input_sha256"]:
        raise RuntimeError("The input changed during inference.")
    if ENABLE_GRADCAM and any("gradcam" not in row for row in pending_result["predictions"]):
        raise RuntimeError("Grad-CAM was requested but its artifacts were not recorded.")
    artifact_paths = gradcam_artifact_paths(pending_result)
    artifact_records = [{"filename": path.name, "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                         "size_bytes": path.stat().st_size} for path in artifact_paths]
    if artifact_paths:
        cli_artifacts = pending_result["metadata"]["gradcam"].get("artifacts", [])
        if sorted(cli_artifacts, key=lambda item: item["filename"]) != sorted(artifact_records, key=lambda item: item["filename"]):
            raise RuntimeError("The visualization files do not match the CLI's saved artifact hashes.")
    # Link the output to its run record without storing an absolute input path.
    run_record = {
        "code_commit": CODE_COMMIT,
        "source_ref": GIT_REF,
        "source_sha256": SOURCE_SHA256,
        "input": input_record,
        "prediction_file": OUTPUT_JSON.name,
        "prediction_sha256": hashlib.sha256(OUTPUT_JSON.read_bytes()).hexdigest(),
        "runtime": runtime_versions,
        "inference_versions": pending_result["metadata"]["versions"],
        "actual_device": pending_result["metadata"]["actual_device"],
        "gradcam": pending_result["metadata"].get("gradcam"),
        "artifacts": artifact_records,
    }
    from takotsubo_ami_ai_ecg.safe_output import write_files
    run_bytes = (json.dumps(run_record, ensure_ascii=False, indent=2, allow_nan=False) + "\n").encode("utf-8")
    write_files([(RUN_JSON, run_bytes)], protected_paths=[IMAGE_PATH, OUTPUT_JSON, *required_weights, *artifact_paths])
    result = pending_result
    render_progress("Inference completed; prediction and run record saved.")
except BaseException as exc:
    result = None
    try:
        stop_inference_process()
    finally:
        failure_message = "Inference cancelled." if isinstance(exc, KeyboardInterrupt) else "Inference failed. Review the error below."
        try:
            render_progress(failure_message)
        except Exception:
            print(failure_message, flush=True)
    raise
finally:
    if process is not None and process.stdout is not None:
        process.stdout.close()

print("Inference completed.")
print("Prediction:", OUTPUT_JSON.name)
print("Run record:", RUN_JSON.name)


The next cells show the saved prediction and optional Grad-CAM overlays. The final download cell creates **one ZIP containing the prediction JSON, matching run-record JSON and all visualization artifacts from that run**. With Grad-CAM disabled, it contains only the two JSON files. With Grad-CAM enabled, it also contains a prepared input-image copy; keep this ZIP private when the input is private. Model weights are never included.


## 7. Review the results

The class order is **0 = tako, 1 = AMI**. By default, the ensemble uses `prob_AMI >= 0.5`, while individual models use argmax (a probability tie is assigned to AMI for the ensemble and tako for an individual model). If you set a numeric `THRESHOLD`, it is used for either case. The following cell displays model outputs. Without comparison against ground-truth labels, these outputs do not establish sensitivity, specificity, or accuracy.

The JSON records the actual execution settings, weight identifiers, logits, and probabilities. Colab's working storage is temporary, so save any results you need to retain.
Successful inference creates `prediction_<model>_<timestamp>_<id>.json` and a matching file ending in `.colab_run.json`. The latter records the code commit, runtime environment, and SHA-256 of the prediction file. The run record also links visualization files by their names and SHA-256. Use the download cell below to save the complete run.


In [ ]:
if not isinstance(globals().get("result"), dict):
    raise RuntimeError("Complete inference successfully before reviewing the results.")

for prediction in result["predictions"]:
    summary = {
        "sample_id": prediction.get("sample_id"),
        "pred_id": prediction["pred_id"],
        "pred_label": prediction["pred_label"],
        "logits": prediction["logits"],
        "probabilities": prediction["probabilities"],
        "models": prediction.get("models", {}),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))


## 8. Review Grad-CAM

These maps show **relative positive contributions to one class logit on the unaugmented image**. Prediction probabilities above may include TTA; the heatmap is not an explanation of the entire TTA aggregation. Its color value is not a class probability or a calibrated importance score, and maps from different models are scaled independently.

The blue-to-yellow palette runs from lower to higher map values. A constant map is shown in neutral gray and its overlay leaves the image unchanged. Each panel identifies the selected class and normalization. The optional `legacy-vit` setting is for comparison with the old ViT preprocessing; it is not a claim that the historical reader-study images have been reproduced.


In [ ]:
if not isinstance(globals().get("result"), dict):
    raise RuntimeError("Complete inference successfully before viewing Grad-CAM.")

import base64
from html import escape
from IPython.display import HTML, display

saved_record = json.loads(RUN_JSON.read_text(encoding="utf-8"))
if saved_record.get("prediction_sha256") != hashlib.sha256(OUTPUT_JSON.read_bytes()).hexdigest():
    raise RuntimeError("The saved prediction changed after the run record was created.")
saved_result = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
paths = gradcam_artifact_paths(saved_result)
expected_artifacts = {item["filename"]: item for item in saved_record.get("artifacts", [])}
for path in paths:
    item = expected_artifacts.get(path.name, {})
    data = path.read_bytes()
    if item.get("sha256") != hashlib.sha256(data).hexdigest() or item.get("size_bytes") != len(data):
        raise RuntimeError("A visualization file does not match its saved run record.")

def png_panel(filename, title):
    data = (OUTPUT_JSON.parent / filename).read_bytes()
    encoded = base64.b64encode(data).decode("ascii")
    return (f'<figure style="margin:0;max-width:480px"><figcaption>{escape(title)}</figcaption>'
            f'<img alt="{escape(title)}" src="data:image/png;base64,{encoded}" '
            'style="width:100%;height:auto" /></figure>')

for prediction in saved_result["predictions"]:
    cam = prediction.get("gradcam")
    if cam is None:
        print("Grad-CAM was disabled for this run.")
        continue
    display(HTML(png_panel(cam["input_image"], "Prepared input (480 x 480)")))
    for model, details in cam["models"].items():
        heading = f"{model}: target {details['target_label']} ({details['target_source']})"
        print(heading)
        print("Unaugmented logits [tako, AMI]:", details["unaugmented_logits"])
        print("Normalization:", details["normalization"])
        if details.get("map_is_constant"):
            print("Constant map: no spatial ranking is available; the overlay is unchanged.")
        panels = png_panel(details["files"]["heatmap"], "Relative Grad-CAM: low (blue) to high (yellow)")
        panels += png_panel(details["files"]["overlay"], f"{model} overlay; target {details['target_label']}")
        display(HTML('<div style="display:flex;flex-wrap:wrap;gap:12px">' + panels + '</div>'))


In [ ]:
# Download this run's linked JSON records and optional visualizations as one ZIP.
DOWNLOAD_RESULTS = True
if DOWNLOAD_RESULTS:
    import hashlib
    import json
    from io import BytesIO
    from uuid import uuid4
    from zipfile import ZIP_DEFLATED, ZipFile
    from google.colab import files
    from takotsubo_ami_ai_ecg.safe_output import write_files

    if not isinstance(globals().get("result"), dict) or not OUTPUT_JSON.is_file() or not RUN_JSON.is_file():
        raise RuntimeError("Confirm that inference completed and both JSON files were saved.")
    if any(p.is_symlink() for path in (OUTPUT_JSON, RUN_JSON) for p in (path, *path.parents)):
        raise RuntimeError("Result files must not use symbolic links.")
    prediction_bytes = OUTPUT_JSON.read_bytes()
    run_bytes = RUN_JSON.read_bytes()
    record = json.loads(run_bytes)
    if (record.get("prediction_file") != OUTPUT_JSON.name
            or record.get("prediction_sha256") != hashlib.sha256(prediction_bytes).hexdigest()):
        raise RuntimeError("The prediction and run record do not match. Check the selected files.")
    saved_prediction = json.loads(prediction_bytes)
    artifact_paths = gradcam_artifact_paths(saved_prediction)
    artifacts = record.get("artifacts", [])
    if (not isinstance(artifacts, list) or len(artifacts) != len(artifact_paths)
            or [item.get("filename") for item in artifacts] != [path.name for path in artifact_paths]):
        raise RuntimeError("The visualization files do not match the run record.")
    contents = [(OUTPUT_JSON.name, prediction_bytes), (RUN_JSON.name, run_bytes)]
    for path, item in zip(artifact_paths, artifacts):
        data = path.read_bytes()
        if (hashlib.sha256(data).hexdigest() != item.get("sha256")
                or len(data) != item.get("size_bytes")):
            raise RuntimeError("A visualization file changed after the run record was saved.")
        contents.append((path.name, data))

    with BytesIO() as buffer:
        with ZipFile(buffer, "w", compression=ZIP_DEFLATED) as archive:
            for name, data in contents:
                archive.writestr(name, data)
        zip_bytes = buffer.getvalue()

    ZIP_PATH = OUTPUT_JSON.parent / f"{OUTPUT_JSON.stem}_{uuid4().hex[:8]}.zip"
    write_files([(ZIP_PATH, zip_bytes)], protected_paths=[OUTPUT_JSON, RUN_JSON, *artifact_paths])
    print("ZIP prepared:", ZIP_PATH.name)
    print("Included:", ", ".join(name for name, _ in contents))
    files.download(str(ZIP_PATH))


## Validation scope

The v0.2.1 prediction baseline passed a WSL/RTX 5090 float32 no-TTA comparison on ten private ECG images, with no changed classifications and all outputs within the recorded tolerance. Its final public-tag Colab/A100 runs also completed the synthetic-image example with paper TTA; source, selected-weight identities and saved result records were verified. Those baseline checks are separate from the Grad-CAM checks below.

On 2026-10-23, Grad-CAM candidate `0.3.0.dev0` completed 119 local Windows CPU tests: 116 passed and three hard-link tests were skipped because the selected temporary directory did not support them. Using the three selected checkpoints and one generated 480 x 480 image, paired CPU float32 checks passed without TTA and with paper TTA (ViT 24/magnitude 1, ConvNeXtV2 1, MLP-Mixer 32/magnitude 3). A paired WSL/RTX 5090 CUDA float32 check also passed with paper TTA, Python 3.12.12, batch size 1 and zero workers. In each paired check, enabling Grad-CAM left all prediction values exactly unchanged.

The candidate also completed Colab/A100 CUDA float32 paper-TTA inference and Grad-CAM on one generated image, using Python 3.13.15, batch size 1 and four workers. Both GPU workflows produced two finite float32 480 x 480 maps with verified PNG/NPY artifacts. The Colab candidate archive and all 111 recorded source hashes matched; its downloaded ZIP contained the matching two JSON records and seven visualization artifacts. This Colab run did not include a paired Grad-CAM-off comparison.

The WSL and Colab GPU checks used torch 2.13.0+cu130, torchvision 0.28.0+cu130, transformers 5.10.4 and Pillow 13.0.0. Release `0.3.0` differed from the tested candidate only in documentation and package-version metadata.

On 2026-10-24, the published `v0.3.0` workflow completed paper-TTA inference and Grad-CAM on Colab/A100 using Python 3.13.16, the same core inference versions, CUDA float32, one generated 480 x 480 image, batch size 1 and four workers. The run recorded commit `341ac955353f535fb74a4b6d8005a8b3b1290ad4`; all 20 recorded critical source hashes, the three official checkpoint hashes and the ViT configuration hash matched. Independent recalculation confirmed each model's softmax and the equal-weight raw-logit ensemble. Both maps were finite float32 480 x 480 arrays, and all nine saved files (two JSONs, five PNGs and two NPYs) passed integrity and correspondence checks. This public Colab run did not include a paired Grad-CAM-off comparison. See [implementation status](https://github.com/ekagawa007/takotsubo_ami_ai_ecg/blob/main/docs/implementation_status.md) for the complete validation record.

On 2026-10-28, the `v0.3.1` candidate completed WSL/RTX 5090 checks with Python 3.12.12 and filelock 4.2.1: 117 of 119 tests passed and two Windows-only cases were skipped. With one generated image, paper TTA, CUDA float32, batch 1 and zero workers, the filelock substitution preserved all prediction values and both numerical Grad-CAM maps exactly. The original environment was retained.

The matching candidate archive also completed this workflow in a fresh Colab/A100 inference environment with Python 3.13.16, CUDA float32, filelock 4.2.1, paper TTA, batch 1 and four workers. Its 112 source hashes, selected weights/configuration, synthetic RGB pixels, prediction calculation and all nine output files were verified. Both finite float32 480 x 480 maps matched their saved heatmaps and overlays. This Colab run did not include a paired Grad-CAM-off comparison. The public notebook uses Git checkout in place of candidate archive loading; all subsequent code cells are unchanged. Final public-tag source acquisition is checked separately after publication. These candidate results do not establish historical paper-TTA or reader-study visualization equivalence.

Grad-CAM uses separate unaugmented float32 passes and does not explain the complete TTA average or three-model ensemble. The layer and gradient calculations follow the archived two-model visualization code, with current-inference normalization by default. ViT legacy preprocessing, display palette and raster export differ as documented in `docs/gradcam.md`. Historical reader-study heatmap equivalence remains unverified.

These software checks do not establish clinical performance or numerical equivalence to historical stochastic paper-TTA runs. TTA results depend on image order, batch size, worker count, library versions and precision.

The run record identifies the source, critical source files, input hashes, actual inference environment and exported visualization hashes. It does not contain the absolute input path. Keep records and image-containing result ZIPs private when their associated images are private, and clear notebook outputs before sharing an executed notebook.
